In [1]:
#Imports

from __future__ import annotations

import json
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import unary_union

import osmnx as ox
import os
from osm_source import OSM_SOURCE, load_osm_features

pd.set_option('display.max_columns', None)  

In [2]:
territory = "GG" #GE ou GG
network = "walk"  # bike ou walk

#Load paths

# Sorties
if territory == "GG":
    output_dir = Path("../../Data/input/networkGG")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    gpkg_attr = Path("../../Data/input/attributs/GG/osm/osm_attributes.gpkg")  # for GG
    gpkg_attr.parent.mkdir(parents=True, exist_ok=True)

elif territory == "GE":
    output_dir = Path("../../Data/input/networkGE")
    output_dir.mkdir(parents=True, exist_ok=True)

    gpkg_attr = Path("../../Data/input/attributs/GE/osm/osm_attributes.gpkg")  # for GG
    gpkg_attr.parent.mkdir(parents=True, exist_ok=True)

# Zone d'étude
area_mode = "shapefile"  # "shapefile" | "geocode"
shapefile_path = Path("../../Data/input/network_agreg/AGGLO_PERIMETRE_AVEC_LAC-SHP/AGGLO_PERIMETRE_AVEC_LAC.shp")
places = ["Genève, Switzerland", "Vandoeuvres, Switzerland", "Cologny, Switzerland", "Collonge-Bellerive, Switzerland", "Choulex, Switzerland"]  # si geocode

# Exports
export_nodes = True
export_edges = True
export_graphml = True
export_formats = ["parquet", "geojson"]  # "parquet" | "geojson" | "gpkg"

# CRS · séparer opérations métriques et export final
operation_crs = "EPSG:2056"  # CRS métrique pour calculs (distances, buffers, etc.)
export_crs = "EPSG:4326"     # CRS final pour les exports

if network == "bike":
    edges_gdf_graph = gpd.read_file(output_dir / "bike_edges_graph.geojson")
    nodes_gdf = gpd.read_file(output_dir / "bike_nodes_graph.geojson")

if network == "walk":
    edges_gdf_graph = gpd.read_file(output_dir / "walk_edges_graph.geojson")
    nodes_gdf = gpd.read_file(output_dir / "walk_nodes_graph.geojson")


conflict_zones = gpd.read_file(output_dir / "pedestrian_bike_conflicts.geojson")



## Fonctions utilitaires (simples et réutilisables)

In [3]:
def load_aoi(area_mode: str, shapefile_path: Path, places: list[str]) -> gpd.GeoDataFrame:
    """Retourne une GeoDataFrame (EPSG:4326) contenant un polygone d'AOI."""
    if area_mode == "shapefile":
        gdf = gpd.read_file(shapefile_path)
        gdf = gdf.to_crs("EPSG:4326") if gdf.crs is not None else gdf.set_crs("EPSG:4326")
        geom = unary_union(gdf.geometry.values)
        return gpd.GeoDataFrame({"name":["aoi"]}, geometry=[geom], crs="EPSG:4326")

    if area_mode == "geocode":
        polys = []
        for p in places:
            g = ox.geocode_to_gdf(p).to_crs("EPSG:4326")
            polys.append(unary_union(g.geometry.values))
        geom = unary_union(polys)
        return gpd.GeoDataFrame({"name":["aoi"]}, geometry=[geom], crs="EPSG:4326")

    raise ValueError(f"area_mode inconnu: {area_mode}")


def to_crs(gdf: gpd.GeoDataFrame, crs: str) -> gpd.GeoDataFrame:
    """Convertit vers le CRS spécifié en gérant les cas sans CRS défini."""
    if gdf.crs is None:
        return gdf.set_crs("EPSG:4326").to_crs(crs)
    return gdf.to_crs(crs)


def to_metric_crs(gdf: gpd.GeoDataFrame, metric_crs: str) -> gpd.GeoDataFrame:
    """Convertit vers un CRS métrique pour les calculs de distance."""
    return to_crs(gdf, metric_crs)


def to_export_crs(gdf: gpd.GeoDataFrame, export_crs: str) -> gpd.GeoDataFrame:
    """Convertit vers le CRS d'export final."""
    return to_crs(gdf, export_crs)


def explode_lines(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf[gdf.geometry.notna()].copy()
    return gdf.explode(index_parts=False, ignore_index=True)


def simplify_gdf(gdf: gpd.GeoDataFrame, tol_m: float, metric_crs: str) -> gpd.GeoDataFrame:
    """Simplifie en métrique puis reprojette dans le CRS d'origine."""
    if tol_m and tol_m > 0:
        original_crs = gdf.crs
        gdf_metric = to_metric_crs(gdf, metric_crs)
        gdf_metric = gdf_metric.copy()
        gdf_metric["geometry"] = gdf_metric.geometry.simplify(tol_m, preserve_topology=True)
        return to_crs(gdf_metric, str(original_crs))
    return gdf

import math
import numpy as np
from shapely.geometry import LineString

def mean_bearing(line: LineString) -> float:
    """
    Calcule une orientation moyenne simple d'une LineString (en radians).
    Approche volontairement robuste et peu coûteuse.
    """
    if line is None or line.is_empty:
        return np.nan

    coords = list(line.coords)
    if len(coords) < 2:
        return np.nan

    x0, y0 = coords[0]
    x1, y1 = coords[-1]

    dx = x1 - x0
    dy = y1 - y0

    if dx == 0 and dy == 0:
        return np.nan

    return math.atan2(dy, dx)


def safe_object_columns_for_parquet(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Normalise les colonnes object mixtes (ex. osmid liste/scalar) pour éviter les erreurs parquet."""
    out = gdf.copy()
    for col in out.columns:
        if col == out.geometry.name:
            continue
        if out[col].dtype == "object":
            has_list = out[col].apply(lambda v: isinstance(v, (list, tuple, set))).any()
            if has_list:
                out[col] = out[col].apply(
                    lambda v: None if pd.isna(v) else json.dumps(list(v)) if isinstance(v, (list,tuple,set)) else json.dumps([v])
                )
            else:
                out[col] = out[col].apply(lambda v: None if pd.isna(v) else str(v))
    return out

def col_series(df, col, default=None):
    # default sert si tu veux une série de valeurs par défaut, sinon NaN
    if col in df.columns:
        return df[col]
    if default is None:
        return pd.Series(pd.NA, index=df.index)
    return pd.Series(default, index=df.index)

def col_notna(df, col):
    if col in df.columns:
        return df[col].notna()
    return pd.Series(False, index=df.index)

def col_isin(df, col, values):
    if col in df.columns:
        return df[col].isin(values)
    return pd.Series(False, index=df.index)

def col_eq(df, col, value):
    if col in df.columns:
        return df[col].eq(value)
    return pd.Series(False, index=df.index)



In [4]:

def keep_columns(gdf: gpd.GeoDataFrame, keep_cols: list[str]) -> gpd.GeoDataFrame:
    cols = ["geometry", "osmid"] + [c for c in keep_cols if c in gdf.columns]
    cols = list(dict.fromkeys(cols))  # unique, stable
    return gdf[cols].copy()

def true_tag():
    """
    Pour ox.features_from_polygon: passer True récupère 'tag exists' dans OSMnx
    (ça marche généralement), sinon tu peux élargir avec une liste de valeurs.
    """
    return True

In [5]:
aoi = load_aoi(area_mode, shapefile_path, places)  # ta fonction existe déjà
aoi_poly = aoi.geometry.iloc[0]

In [6]:
# ox.settings.requests_timeout = 600  # seconds
# # ox.settings.overpass_url = "https://overpass.kumi.systems/api"   # OSMnx >= 2.0 (sans /interpreter)
# # Alternative si kumi ne répond pas :
# ox.settings.overpass_url = "https://overpass.private.coffee/api"

In [7]:
ox.settings.requests_timeout = 600  # seconds (valeur restaurée après chaque fetch)
# Le serveur Overpass est géré par fetch_osm_features_with_retry (liste OVERPASS_ENDPOINTS).
# Optionnel, si les miroirs imposent de longues pauses via /status :
# ox.settings.overpass_rate_limit = False

aoi = load_aoi(area_mode, shapefile_path, places)  # ta fonction existe déjà
aoi_poly = aoi.geometry.iloc[0]


In [8]:
# import fiona

# # 1) La couche existe-t-elle dans le GeoPackage ?
# print(fiona.listlayers(gpkg_attr))

# # 2) + 3) Extraction brute, sans filtre géométrique ni try/except
# raw = ox.features_from_polygon(
#     aoi_poly, {"highway": "pedestrian", "area:highway": "pedestrian"}
# ).reset_index()
# print(len(raw))
# print(raw.geom_type.value_counts())

# # Répartition géométrie x type de tag
# raw["source_tag"] = np.where(raw.get("area:highway").eq("pedestrian"), "area:highway", "highway")
# print(pd.crosstab(raw["source_tag"], raw.geom_type))

## 2. Get features

1. POI
2. Amenités
3. Edges (déjà présents, simple filtre)
4. Nodes (déjà présents, simple filtre)

In [9]:
import time

try:
    from osmnx._errors import InsufficientResponseError   # levée par OSMnx quand aucune entité ne correspond
except ImportError:
    InsufficientResponseError = ()                       # vieille version : on ne distingue pas ce cas

OVERPASS_ENDPOINTS = [
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api",
    "https://overpass-api.de/api",
]

MIN_COLS = ["geometry", "osmid"]
OSM_FAILURES = []   # couches dont l'extraction a échoué pendant le run


def fetch_osm_features_with_retry(
    polygon,
    tags,
    label: str,
    *,
    max_attempts: int = 4,
    timeout_s: int = 600,
    base_pause_s: float = 6.0,
    endpoints: list[str] | None = None,
    geometry_types: set[str] | None = None,
) -> gpd.GeoDataFrame:



    if OSM_SOURCE == "local":
        print(f"[{label}] lecture locale", flush=True)
        try:
            return load_osm_features(polygon, tags, territory, geometry_types, variant="full")
        except InsufficientResponseError:
            print(f"[{label}] aucune entité OSM correspondante.")
            return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs="EPSG:4326")
        except Exception as exc:
            raise RuntimeError(
                f"Lecture locale échouée pour '{label}' : {type(exc).__name__}: {exc}"
            ) from exc

    """Télécharge des features OSM avec retries + alternance des serveurs Overpass.
    Lève RuntimeError si toutes les tentatives échouent."""
    endpoints = endpoints or OVERPASS_ENDPOINTS

    # OSMnx >= 2.0 : overpass_url ; OSMnx 1.x : overpass_endpoint (même format, sans /interpreter)
    endpoint_attr = "overpass_url" if hasattr(ox.settings, "overpass_url") else "overpass_endpoint"
    previous_endpoint = getattr(ox.settings, endpoint_attr, None)
    previous_timeout = ox.settings.requests_timeout
    last_error = None

    try:
        for attempt in range(1, max_attempts + 1):
            endpoint = endpoints[(attempt - 1) % len(endpoints)]
            setattr(ox.settings, endpoint_attr, endpoint)
            ox.settings.requests_timeout = timeout_s

            try:
                gdf = ox.features_from_polygon(polygon, tags).reset_index()
            except InsufficientResponseError:
                # Requête aboutie mais aucune entité : pas une panne, inutile de réessayer
                print(f"[{label}] aucune entité OSM correspondante.")
                return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs="EPSG:4326")
            except Exception as exc:
                last_error = exc
                print(f"[{label}] tentative {attempt}/{max_attempts} échouée sur {endpoint}: {type(exc).__name__}.")
                if attempt < max_attempts:
                    wait_s = base_pause_s * (2 ** (attempt - 1))
                    print(f"[{label}] nouvelle tentative dans {wait_s:.0f}s...")
                    time.sleep(wait_s)
                continue

            gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs="EPSG:4326")
            if geometry_types:
                gdf = gdf[gdf.geometry.type.isin(sorted(geometry_types))].copy()
            return gdf

        raise RuntimeError(
            f"Échec du téléchargement OSM pour '{label}' après {max_attempts} tentatives."
        ) from last_error

    finally:
        if previous_endpoint is not None:
            setattr(ox.settings, endpoint_attr, previous_endpoint)
        ox.settings.requests_timeout = previous_timeout


def extract_osm_features(aoi_poly, tags: dict, geom_types=("Point",), keep_cols=None, label=None):
    keep_cols = keep_cols or []
    label = label or str(tags)
    empty = gpd.GeoDataFrame(columns=MIN_COLS + keep_cols, geometry="geometry", crs="EPSG:4326")

    try:
        gdf = fetch_osm_features_with_retry(aoi_poly, tags, label=label, geometry_types=set(geom_types))
    except RuntimeError as e:
        print(f"❌ {e}")
        OSM_FAILURES.append(label)
        return empty

    if gdf.empty:
        return empty

    gdf = gdf[gdf.geometry.notna()].copy()

    # ---- Assurer une colonne osmid ----
    if "osmid" not in gdf.columns:
        if "id" in gdf.columns:
            gdf["osmid"] = gdf["id"]
        elif "osm_id" in gdf.columns:
            gdf["osmid"] = gdf["osm_id"]
        elif "@id" in gdf.columns:
            gdf["osmid"] = gdf["@id"]
        elif "index" in gdf.columns:
            gdf["osmid"] = gdf["index"]
        else:
            gdf["osmid"] = range(len(gdf))

    cols = ["geometry", "osmid"] + [c for c in keep_cols if c in gdf.columns]
    gdf = gdf[cols].copy()
    return gdf.set_crs("EPSG:4326", allow_override=True)

In [10]:
from pathlib import Path

def export_layer_gpkg(gdf, gpkg_path: Path, layer: str, export_crs="EPSG:4326"):
    """Écrit/remplace UNE couche du gpkg ; les autres couches sont conservées."""
    if gdf is None or len(gdf) == 0:
        return
    gdf.to_crs(export_crs).to_file(gpkg_path, layer=layer, driver="GPKG", mode="w")


In [11]:
POI_AMENITES_DEFINITIONS = {
    "amenite": {
        "tags": {"amenity": True},
        "keep_cols": ["amenity", "name", "operator", "access"]
    }
}



In [12]:
POI_PEDESTRIAN_DEFINITIONS = {
    # fontaines / eau potable
    "fontaine": {
        "tags": {"amenity": ["drinking_water", "fountain"]},
        "keep_cols": ["amenity", "name", "operator", "access"]
    },

    # amenités “piéton”
    "banc": {
        "tags": {"amenity": "bench"},
        "keep_cols": ["amenity", "backrest", "covered"]
    },
    "toilette": {
        "tags": {"amenity": "toilets"},
        "keep_cols": ["amenity", "access", "wheelchair", "fee", "opening_hours"]
    }
}


In [13]:
POI_TP_DEFINITION = {

    "transport_public": {
        "tags": {
            # Ancien schéma
            "highway": ["bus_stop"],
            "railway": ["station", "halt", "tram_stop"],
            "amenity": ["bus_station"],

            # Schéma moderne OSM
            "public_transport": ["station"]
        },
        "keep_cols": [
            "highway", "railway", "amenity", "public_transport",
            "name", "operator", "network", "ref",
            "wheelchair", "shelter", "bench"
        ]
    }
}


In [14]:
OSM_NODE_FEATURES_PEDESTRIAN = {
    "crossing": {
        "tags": {"crossing": true_tag()},
        "keep_cols": ["crossing", "crossing:markings", "crossing:signals"]
    },
    "traffic_signals": {
        "tags": {"highway": "traffic_signals"},
        "keep_cols": ["highway", "button_operated", "tactile_paving"]
    },
    "barrier": {
        "tags": {"barrier": true_tag()},
        "keep_cols": ["barrier", "access", "bicycle", "wheelchair"]
    },
    "traffic_calming_point": {
        "tags": {"traffic_calming": true_tag()},
        "keep_cols": ["traffic_calming"]
    },
}


In [15]:
WALK_CHEMIN_HW = ["path", "track"]

OSM_WAY_FEATURES_PEDESTRIAN = {

    # type de chemin / niveau de confort
    "chemin": {
        "tags": {"highway": WALK_CHEMIN_HW},
        "keep_cols": ["highway", "tracktype", "surface"]
    },


    "eclairage": {"tags": {"lit": true_tag()}, "keep_cols": ["lit"]},

    # pente OSM (indicatif) — pour une pente fiable, mieux: DEM + sampling
    "topographie_osm": {"tags": {"incline": true_tag()}, "keep_cols": ["incline"]},
}


In [16]:
OSM_POLY_FEATURES_PEDESTRIAN = {
    "canopee": {
        # proxy canopée : zones boisées + alignements/arbres (arbres = points, ici on prend polygones)
        "tags": {"landuse": ["forest"], "natural": ["wood"]},
        "keep_cols": ["landuse", "natural", "name"]
    },
    "espaces_ouverts": {
        "tags": {
            "leisure": ["park", "playground"],
            "landuse": ["recreation_ground"],
            "amenity": ["grave_yard"],  # souvent public et vert
            "place":["islet", "square"]
        },
        "keep_cols": [
            "leisure", "landuse", "amenity",
            "name", "access", "operator", "ownership"
        ]
    },
        "zone_pietonne_area": {
        "tags": {"highway": ["pedestrian"], "area:highway": ["pedestrian"]},
        "keep_cols": ["highway", "area", "area:highway", "name", "access"]
},
    "lac_cours_deau": {
        "tags": {"natural": ["water"], "waterway": ["riverbank"]},
        "keep_cols": ["natural", "water", "name", "waterway"]
    }
}


In [17]:
EDGE_ATTRIBUTES_PEDESTRIAN = {

    "vitesse": {
        "filter": lambda e: (
            col_notna(e, "speed_ctx_kmh")          # vitesse de contexte (0_1, section 7)
            | col_notna(e, "maxspeed")
            | col_notna(e, "maxspeed:forward")
            | col_notna(e, "maxspeed:backward")
            | col_notna(e, "zone:maxspeed")
            | col_notna(e, "maxspeed:type")
        ),
        "keep_cols": [
            "speed_ctx_kmh",                        # variable à utiliser dans l'indice
            "walk_class",                           # proxy (tag propre) ou trottoir/traversée (chaussée adjacente)
            "adj_osm_id", "adj_highway",            # chaussée retenue
            "adj_speed_kmh", "adj_dist_m",          # sa vitesse et sa distance au segment
            "highway",
            "maxspeed", "maxspeed:forward", "maxspeed:backward",
            "zone:maxspeed", "maxspeed:type",
            "traffic_calming", "living_street",
            "name",
        ],
    },

    "zone_pietonne_edge": {
        "filter": lambda e: (
            col_eq(e, "highway", "pedestrian")
            | col_eq(e, "area:highway", "pedestrian")
        ),
        "keep_cols": ["highway", "area:highway", "access", "motor_vehicle"],
    },

    "chemin_edge": {
        "filter": lambda e: col_isin(e, "highway", ["footway", "pedestrian", "steps", *WALK_CHEMIN_HW]),
        "keep_cols": ["highway", "footway", "surface", "smoothness", "lit", "incline", "segregated", "tracktype"],
    },
}

In [18]:
def keep_columns(gdf, cols):
    out_cols = ["geometry"]
    # ajouter un id si dispo
    for id_col in ["osmid", "id", "fid", "edge_id"]:
        if id_col in gdf.columns:
            out_cols.append(id_col)
            break
    out_cols += [c for c in cols if c in gdf.columns]
    out_cols = list(dict.fromkeys(out_cols))  # unique
    return gdf[out_cols].copy()


In [19]:


# ox.settings.requests_timeout = 600  # seconds
# ox.settings.overpass_url = "https://overpass.kumi.systems/api"   # OSMnx >= 2.0 (sans /interpreter)
# # Alternative si kumi ne répond pas :
# # ox.settings.overpass_url = "https://overpass.private.coffee/api"

In [20]:
import time
from datetime import datetime

# Affiche les messages internes d'OSMnx : pause en attente d'un créneau Overpass,
# requête envoyée, réponse lue depuis le cache…
ox.settings.log_console = True

OSM_FAILURES.clear()

# --- POI + Node features (points)
ALL_POI = {}
ALL_POI.update(POI_PEDESTRIAN_DEFINITIONS)
ALL_POI.update(OSM_NODE_FEATURES_PEDESTRIAN)
ALL_POI.update(POI_AMENITES_DEFINITIONS)

N_OVERPASS = (len(ALL_POI) + len(OSM_WAY_FEATURES_PEDESTRIAN)
              + len(OSM_POLY_FEATURES_PEDESTRIAN) + len(POI_TP_DEFINITION))
_counter = 0
T_START = time.perf_counter()


def _now():
    return datetime.now().strftime("%H:%M:%S")


def extract_timed(layer, cfg, geom_types):
    """Extraction Overpass avec affichage du début, de la fin et de la durée."""
    global _counter
    _counter += 1
    print(f"\n[{_now()}] ({_counter}/{N_OVERPASS}) {layer} → extraction en cours…", flush=True)
    t0 = time.perf_counter()
    gdf = extract_osm_features(aoi_poly, cfg["tags"], geom_types=geom_types,
                               keep_cols=cfg["keep_cols"], label=layer)
    dt = time.perf_counter() - t0
    status = "❌ ÉCHEC" if layer in OSM_FAILURES else f"{len(gdf):,} entités"
    elapsed = (time.perf_counter() - T_START) / 60
    print(f"[{_now()}] ({_counter}/{N_OVERPASS}) {layer} : {status} en {dt:.1f}s "
          f"| total écoulé {elapsed:.1f} min", flush=True)
    return gdf


def export_timed(gdf, layer, cfg):
    """Export gpkg avec affichage ; signale explicitement quand rien n'est écrit."""
    if gdf.empty:
        print(f"   ⚠️ {layer} : rien à exporter (une ancienne version peut subsister dans le gpkg)", flush=True)
        return
    t0 = time.perf_counter()
    export_layer_gpkg(keep_columns(gdf, cfg["keep_cols"]), gpkg_attr, layer, export_crs)
    print(f"   💾 {layer} exportée ({time.perf_counter() - t0:.1f}s)", flush=True)


print(f"[{_now()}] Début de l'extraction : {N_OVERPASS} couches Overpass à traiter", flush=True)

# --- POI + Node features (points)
print("\n========== POINTS ==========", flush=True)
for layer, cfg in ALL_POI.items():
    gdf = extract_timed(layer, cfg, ("Point",))
    export_timed(gdf, layer, cfg)

# --- Ways features (lignes)
print("\n========== LIGNES ==========", flush=True)
for layer, cfg in OSM_WAY_FEATURES_PEDESTRIAN.items():
    gdf = extract_timed(layer, cfg, ("LineString", "MultiLineString"))
    export_timed(gdf, layer, cfg)

# --- Polygones
print("\n========== POLYGONES ==========", flush=True)
for layer, cfg in OSM_POLY_FEATURES_PEDESTRIAN.items():
    gdf = extract_timed(layer, cfg, ("Polygon", "MultiPolygon"))
    export_timed(gdf, layer, cfg)

# --- Transports publics
print("\n========== TRANSPORTS PUBLICS ==========", flush=True)
for layer, cfg in POI_TP_DEFINITION.items():
    gdf = extract_timed(layer, cfg, ("Point", "Polygon", "MultiPolygon"))
    if not gdf.empty:
        poly_mask = gdf.geometry.type.isin(["Polygon", "MultiPolygon"])
        if poly_mask.any():
            gdf.loc[poly_mask, "geometry"] = gdf.loc[poly_mask].geometry.representative_point()
    export_timed(gdf, layer, cfg)

# --- Edges (depuis ton graphe piéton) : pas d'appel Overpass
print("\n========== COUCHES LOCALES (graphe, sans Overpass) ==========", flush=True)
be = edges_gdf_graph.copy()
for layer, cfg in EDGE_ATTRIBUTES_PEDESTRIAN.items():
    gdf = be[cfg["filter"](be)].copy()
    print(f"[{_now()}] {layer} : {len(gdf):,} arêtes", flush=True)
    export_timed(gdf, layer, cfg)

# --- Conflits
if conflict_zones is not None and not conflict_zones.empty:
    export_layer_gpkg(conflict_zones, gpkg_attr, "conflit_md", export_crs)
    print(f"[{_now()}] conflit_md : {len(conflict_zones):,} entités exportées", flush=True)

# --- Bilan
total_min = (time.perf_counter() - T_START) / 60
print(f"\n[{_now()}] Terminé en {total_min:.1f} min", flush=True)

# --- Contrôle final : un run « propre » ne doit laisser aucun échec
if OSM_FAILURES:
    raise RuntimeError(
        f"Extraction OSM échouée pour {OSM_FAILURES} : ces couches n'ont PAS été mises à jour "
        f"dans {gpkg_attr} (une ancienne version peut subsister). Relancer la cellule."
    )
print("✅ Toutes les couches OSM extraites sans erreur.")

[09:22:28] Début de l'extraction : 16 couches Overpass à traiter

========== POINTS ==========

[09:22:28] (1/16) fontaine → extraction en cours…
[fontaine] lecture locale
[09:23:50] (1/16) fontaine : 1,288 entités en 81.9s | total écoulé 1.4 min
   💾 fontaine exportée (0.0s)

[09:23:50] (2/16) banc → extraction en cours…
[banc] lecture locale
[09:25:10] (2/16) banc : 6,970 entités en 79.9s | total écoulé 2.7 min
   💾 banc exportée (0.0s)

[09:25:10] (3/16) toilette → extraction en cours…
[toilette] lecture locale
[09:26:29] (3/16) toilette : 342 entités en 79.4s | total écoulé 4.0 min
   💾 toilette exportée (0.0s)

[09:26:29] (4/16) crossing → extraction en cours…
[crossing] lecture locale
[09:27:47] (4/16) crossing : 15,031 entités en 78.1s | total écoulé 5.3 min
   💾 crossing exportée (0.1s)

[09:27:47] (5/16) traffic_signals → extraction en cours…
[traffic_signals] lecture locale
[09:29:47] (5/16) traffic_signals : 2,310 entités en 119.9s | total écoulé 7.3 min
   💾 traffic_signals

In [21]:
'''import pandas as pd
import geopandas as gpd

gdf_amenites = gpd.read_file(gpkg_attr, layer="amenite")

vc = gdf_amenites["amenity"].value_counts()

vc.to_csv("amenity_value_counts.csv", header=["count"])'''


'import pandas as pd\nimport geopandas as gpd\n\ngdf_amenites = gpd.read_file(gpkg_attr, layer="amenite")\n\nvc = gdf_amenites["amenity"].value_counts()\n\nvc.to_csv("amenity_value_counts.csv", header=["count"])'